# NB08 — CPTAC protein sanity check (Phase 4 go/no-go)

**Out:** `data/reference/node_reliability.csv`
**Gate:** ≥60% of ODE nodes have RNA↔protein Spearman ≥ 0.3
**If this fails: do not start NB09–NB11 as specified.**


In [ ]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

cwd = Path.cwd().resolve()
for cand in [cwd, *cwd.parents]:
    if (cand / "src" / "gate.py").is_file():
        sys.path.insert(0, str(cand / "src"))
        break
    nested = cand / "v2"
    if (nested / "src" / "gate.py").is_file():
        sys.path.insert(0, str(nested / "src"))
        break

from paths import ensure_src_on_path, resolve_v2_root
from gate import gate as _gate_impl
from safety import assert_safe

V2_ROOT = resolve_v2_root()
ensure_src_on_path(V2_ROOT)
REPO_ROOT = V2_ROOT.parent
RAW = V2_ROOT / "data" / "raw"
INTERIM = V2_ROOT / "data" / "interim"
REF = V2_ROOT / "data" / "reference"
ARTIFACTS = V2_ROOT / "artifacts"
FIGURES = V2_ROOT / "reports" / "figures"
for d in (RAW, INTERIM, REF, ARTIFACTS, FIGURES, INTERIM / "causal_networks"):
    d.mkdir(parents=True, exist_ok=True)

# Laptop vs VPS. Smoke passes are provisional until a full run converts them.
# NB01 and NB04 stay full: harmonisation and the VAE are cheap.
SMOKE_TEST = True
N_SAMPLES  = 200    if SMOKE_TEST else None   # NB02 bulk (BayesPrism; memory)
N_SC_CELLS = 25_000 if SMOKE_TEST else None   # NB02 Wu reference (BayesPrism; memory)
N_PATIENTS = 50     if SMOKE_TEST else None   # NB07 CARNIVAL (throughput, not RAM)
N_DRUGS    = 10     if SMOKE_TEST else None   # NB10 ODE (FLOPs, not RAM)

def gate(*args, **kwargs):
    kwargs.setdefault("smoke_test", SMOKE_TEST)
    return _gate_impl(*args, **kwargs)

print("V2_ROOT =", V2_ROOT, "SMOKE_TEST =", SMOKE_TEST)


In [ ]:
# Config
FRAC_MIN, RHO_MIN = 0.6, 0.3
import numpy as np, pandas as pd
from scipy.stats import spearmanr
nodes = pd.read_csv(REF / "ode_nodes.csv")["gene"].tolist()


In [ ]:
# Load
expr_p, harm_p = INTERIM / "intrinsic_expression.parquet", INTERIM / "harmonised_expression.parquet"
expr = pd.read_parquet(harm_p) if harm_p.exists() else (pd.read_parquet(expr_p) if expr_p.exists() else None)
prot_files = [p for p in (RAW / "cptac_brca").glob("**/*") if p.is_file() and p.name != "PLACEHOLDER.txt" and p.stat().st_size > 1024]
# TCGA pan-can protein quantification is CPTAC packaged by cBioPortal
if not prot_files:
    prot_files = list((RAW / "tcga_brca").rglob("data_protein_quantification.txt")) + list((RAW / "tcga_brca").rglob("data_rppa.txt"))
print("protein files", prot_files)


In [ ]:
# Compute
rhos = {g: float("nan") for g in nodes}
if expr is not None:
    mat = expr.select_dtypes(include=[np.number])
    mat.columns = mat.columns.astype(str).str.upper()
    mat.index = mat.index.astype(str)
    prot = None
    if prot_files:
        f = prot_files[0]
        if f.suffix in {".parquet"}:
            prot = pd.read_parquet(f)
        else:
            prot = pd.read_csv(f, sep="\t", comment="#")
            id_col = "Hugo_Symbol" if "Hugo_Symbol" in prot.columns else prot.columns[0]
            prot = prot.set_index(id_col)
            prot.index = prot.index.astype(str).str.split("|").str[0].str.upper()
            drop = [c for c in prot.columns if "entrez" in c.lower() or "composite" in c.lower()]
            prot = prot.drop(columns=drop, errors="ignore").apply(pd.to_numeric, errors="coerce").T
            prot = prot.T.groupby(level=0).mean().T
        prot.columns = prot.columns.astype(str).str.upper()
        prot.index = prot.index.astype(str)
        # TCGA protein samples are 15-char; expression may be 12-char patients
        prot.index = prot.index.str[:12]
        mat.index = mat.index.str[:12]
    for g in nodes:
        if prot is not None and g in mat.columns and g in prot.columns:
            common = mat.index.intersection(prot.index)
            if len(common) >= 10:
                rhos[g] = float(spearmanr(mat.loc[common, g], prot.loc[common, g]).statistic)
                continue
        rhos[g] = float("nan")
rel = pd.DataFrame({"gene": list(rhos), "spearman_rna_protein": list(rhos.values())})
rel["ok"] = rel["spearman_rna_protein"] >= RHO_MIN
rel["wider_prior"] = ~rel["ok"].fillna(True)
rel.to_csv(REF / "node_reliability.csv", index=False)
finite = rel["spearman_rna_protein"].dropna()
frac = float((finite >= RHO_MIN).mean()) if len(finite) else 0.0
note = f"n_nodes={len(finite)}, median rho={float(finite.median()) if len(finite) else float('nan'):.3f}"
if len(finite) == 0:
    note += " | CPTAC missing — Phase 4 MUST NOT proceed until this gate can be evaluated"
elif not any("cptac" in str(p).lower() for p in prot_files):
    note += " | assay=TCGA_RPPA_not_CPTAC_MS; keep pass; re-run vs CPTAC mass-spec before writeup"
print(rel.to_string(index=False))


In [ ]:
# GATE
gate("NB08", "rna_protein_concordance", float(frac), FRAC_MIN,
     n=int(len(finite)), note=note)


In [ ]:
import matplotlib.pyplot as plt
rel = pd.read_csv(REF / "node_reliability.csv")
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(rel["gene"], rel["spearman_rna_protein"].fillna(0))
ax.axhline(0.3, c="red", ls="--")
plt.xticks(rotation=60, ha="right")
fig.tight_layout(); fig.savefig(FIGURES / "NB08_rna_protein.png", dpi=140)
